# Mapa Interativo de Ciclofaixas do Rio de Janeiro

**Objetivo:** Baixar dados de infraestrutura cicloviária do [OpenStreetMap](https://www.openstreetmap.org/) e criar um mapa interativo usando Python.

### Bibliotecas utilizadas

| Biblioteca | Para que serve |
|---|---|
| `osmnx` | Baixar dados geográficos do OpenStreetMap (ruas, ciclovias, etc.) |
| `geopandas` | Manipular dados geográficos como se fossem tabelas (DataFrames com geometria) |
| `folium` | Criar mapas interativos em HTML (baseado no Leaflet.js) |

### Como funciona

1. Baixamos os dados de ciclovias do Rio de Janeiro via OpenStreetMap
2. Exploramos e classificamos os tipos de via (ciclovia exclusiva, ciclofaixa, via compartilhada)
3. Criamos um mapa interativo com camadas separadas por tipo
4. Salvamos o mapa como arquivo HTML para abrir no navegador

## 1. Instalação e configuração

Rode a célula abaixo para instalar as bibliotecas necessárias. Se já estiverem instaladas, não vai acontecer nada.

In [ ]:
!pip install osmnx geopandas folium

In [ ]:
import os
import osmnx as ox
import geopandas as gpd
import pandas as pd
import folium

# Criar pastas para organizar os arquivos gerados
for pasta in ["dados", "resultados"]:
    os.makedirs(pasta, exist_ok=True)

print("Bibliotecas importadas e pastas criadas com sucesso!")

## 2. Download dos dados do OpenStreetMap

Usamos o **OSMnx** para baixar dados diretamente do OpenStreetMap.

### Por que usamos `features_from_place` em vez de `graph_from_place`?

No OpenStreetMap, a infraestrutura cicloviária está mapeada de **duas formas diferentes**:

1. **Ciclovias exclusivas** — vias próprias, tagueadas como `highway=cycleway`
2. **Ciclofaixas em vias normais** — ruas que têm faixa de bicicleta, tagueadas com `cycleway=lane`, `cycleway=shared_lane`, etc.

A função `graph_from_place` só pega o tipo 1. Para capturar os dois tipos, usamos `features_from_place` que permite buscar por **qualquer tag** do OpenStreetMap.

### O que cada tag significa

| Tag OSM | Significado |
|---|---|
| `highway=cycleway` | Via exclusiva para bicicletas (separada do trânsito) |
| `cycleway=lane` | Ciclofaixa pintada na rua (faixa dedicada, mas na mesma via) |
| `cycleway=track` | Ciclovia separada fisicamente, mas paralela à rua |
| `cycleway=shared_lane` | Via compartilhada com sinalização para bicicletas |
| `cycleway=shared` | Via compartilhada (sem separação) |

> **Nota:** O download pode levar de 30 segundos a 2 minutos dependendo da sua conexão.

In [ ]:
cidade = "Rio de Janeiro, Brazil"

# === FONTE 1: Ciclovias exclusivas (highway=cycleway) ===
# São vias inteiras dedicadas a bicicletas, como a Ciclovia Alfredo Sirkis
print(f"Baixando ciclovias exclusivas de '{cidade}'...")
ciclovias_exclusivas = ox.features_from_place(cidade, tags={"highway": "cycleway"})

# Manter apenas linhas (excluir pontos ou polígonos que podem vir junto)
ciclovias_exclusivas = ciclovias_exclusivas[
    ciclovias_exclusivas.geometry.geom_type == "LineString"
].copy()
ciclovias_exclusivas["tipo_ciclovia"] = "Ciclovia Exclusiva"
print(f"  Encontradas: {len(ciclovias_exclusivas)} vias")

# === FONTE 2: Ciclofaixas em ruas normais (cycleway=lane/track/shared_lane/shared/yes) ===
# São ruas que têm uma faixa ou sinalização para bicicletas
print(f"\nBaixando ciclofaixas em vias normais...")
ciclofaixas = ox.features_from_place(cidade, tags={"cycleway": True})
ciclofaixas = ciclofaixas[
    ciclofaixas.geometry.geom_type == "LineString"
].copy()

# Filtrar apenas os que realmente têm infraestrutura
# (excluir cycleway=no e cycleway=crossing que não são infraestrutura útil)
valores_validos = ["lane", "track", "shared_lane", "shared", "yes"]
ciclofaixas = ciclofaixas[
    ciclofaixas["cycleway"].apply(
        lambda x: str(x) if isinstance(x, list) else x
    ).isin(valores_validos)
].copy()

# Classificar o tipo de ciclofaixa
def classificar_ciclofaixa(valor):
    """Classifica o tipo de ciclofaixa baseado na tag do OpenStreetMap."""
    valor = str(valor)
    if valor in ["lane", "track"]:
        return "Ciclofaixa Dedicada"     # faixa pintada ou separada fisicamente
    else:
        return "Via Compartilhada"        # shared_lane, shared, yes

ciclofaixas["tipo_ciclovia"] = ciclofaixas["cycleway"].apply(classificar_ciclofaixa)
print(f"  Encontradas: {len(ciclofaixas)} vias")

# === COMBINAR AS DUAS FONTES ===
# Preparar colunas em comum
colunas = ["geometry", "tipo_ciclovia", "name"]
for col in colunas:
    if col not in ciclovias_exclusivas.columns:
        ciclovias_exclusivas[col] = None
    if col not in ciclofaixas.columns:
        ciclofaixas[col] = None

infraestrutura_bike = gpd.GeoDataFrame(
    pd.concat([ciclovias_exclusivas[colunas], ciclofaixas[colunas]], ignore_index=True),
    geometry="geometry",
    crs=ciclovias_exclusivas.crs,
)

print(f"\n{'='*50}")
print(f"TOTAL: {len(infraestrutura_bike)} trechos de infraestrutura cicloviária")
print(f"{'='*50}")

## 3. Exploração dos dados

Vamos ver o que baixamos. Cada linha da tabela representa um **trecho de via** com nome, tipo e geometria (coordenadas da linha no mapa).

In [ ]:
# Quantas vias de cada tipo encontramos?
print("Quantidade de vias por tipo:")
print(infraestrutura_bike["tipo_ciclovia"].value_counts())

# Exemplos de nomes de ciclovias conhecidas
nomes = infraestrutura_bike["name"].dropna().unique()
print(f"\nExemplos de vias encontradas ({len(nomes)} com nome):")
for nome in sorted(nomes)[:15]:
    print(f"  - {nome}")

In [ ]:
# Primeiras linhas da tabela
infraestrutura_bike.head(10)

## 4. Preparação para o mapa

Antes de criar o mapa, simplificamos as geometrias para o arquivo HTML não ficar pesado demais.

A simplificação reduz o número de pontos em cada linha, mantendo o formato geral. O parâmetro `tolerance=0.0005` equivale a ~50 metros de precisão — suficiente para visualização.

In [ ]:
# Simplificar geometrias para o mapa não travar no navegador
infraestrutura_bike["geometry"] = infraestrutura_bike["geometry"].simplify(tolerance=0.0005)

# Salvar dados processados como GeoJSON
arquivo_geojson = os.path.join("dados", "ciclovias_rj.geojson")
infraestrutura_bike.to_file(arquivo_geojson, driver="GeoJSON")

print(f"Dados processados salvos em: {arquivo_geojson}")
print(f"Total de trechos: {len(infraestrutura_bike)}")

## 5. Criação do mapa interativo

Usamos o **Folium** para criar um mapa interativo em HTML.

### Conceitos importantes

| Conceito | O que é |
|---|---|
| **Tiles** | O estilo visual do mapa de fundo. Usamos `CartoDB positron` (fundo claro e limpo) |
| **Camadas (layers)** | Cada tipo de ciclovia é uma camada que pode ser ligada/desligada |
| **Tooltip** | Informação que aparece ao passar o mouse sobre uma via |
| **LayerControl** | Botão no canto superior direito para controlar as camadas |

### Cores no mapa

| Tipo | Cor |
|---|---|
| Ciclovia Exclusiva | Vermelho |
| Ciclofaixa Dedicada | Azul |
| Via Compartilhada | Laranja |

In [ ]:
# Configuração: nome da camada → cor no mapa
cores_por_tipo = {
    "Ciclovia Exclusiva":   "red",
    "Ciclofaixa Dedicada":  "blue",
    "Via Compartilhada":    "orange",
}

# Criar o mapa base centrado no Rio de Janeiro
mapa_rj = folium.Map(
    location=[-22.9068, -43.1729],  # Coordenadas do centro do RJ
    zoom_start=12,                   # Nível de zoom (12 = cidade inteira)
    tiles="CartoDB positron",         # Fundo claro e limpo
)

# Adicionar cada tipo de ciclovia como uma camada separada
for tipo, cor in cores_por_tipo.items():
    subset = infraestrutura_bike[infraestrutura_bike["tipo_ciclovia"] == tipo]

    if subset.empty:
        print(f"  Aviso: nenhuma via encontrada para '{tipo}' — pulando.")
        continue

    # Adicionar camada GeoJson ao mapa
    folium.GeoJson(
        subset,
        name=tipo,  # Nome que aparece no controle de camadas
        style_function=lambda x, c=cor: {"color": c, "weight": 2.5},
        tooltip=folium.GeoJsonTooltip(
            fields=["name", "tipo_ciclovia"],
            aliases=["Nome da via:", "Tipo:"],
        ),
    ).add_to(mapa_rj)

    print(f"  Camada '{tipo}': {len(subset)} trechos (cor: {cor})")

# Adicionar controle de camadas (botão no canto superior direito)
folium.LayerControl().add_to(mapa_rj)

# Salvar o mapa como arquivo HTML
arquivo_mapa = os.path.join("resultados", "mapa_interativo_ciclovias.html")
mapa_rj.save(arquivo_mapa)

print(f"\nMapa salvo em: {arquivo_mapa}")
print("Abra esse arquivo no navegador para visualizar o mapa interativo!")

## 6. Visualizar o mapa

Se estiver no **Google Colab** ou **Jupyter Notebook**, o mapa aparece direto abaixo.

Se estiver no **VS Code**, abra o arquivo `resultados/mapa_interativo_ciclovias.html` no navegador.

**Dica:** Use o botão de camadas (canto superior direito) para ligar/desligar cada tipo de ciclovia.

In [ ]:
# Exibir o mapa inline (funciona no Colab e Jupyter Notebook)
mapa_rj